# bge-reranker-v2-m3 리랭킹 실험 v2 — 표 원자성 수정 코퍼스

**v1과 달라진 전제**

| | v1 | v2 |
|---|---|---|
| 코퍼스 | 215청크 (최대 800자) | **181청크 (최대 2,613자)** |
| 파라미터 | alpha .65 · rrf_k 1 · bf .12 · facet .15 | **alpha .65 · rrf_k 4 · bf .06 · facet .05** |
| GT | 원본 | **앵커 보강판** (id 의존도 TEST 5.1% · VAL 3.4%) |
| 후보 깊이 | top-50 | **top-30** |
| 베이스라인 hit@5 | .8186 / .8254 | **.8907 / .8535** |

## 이번에 답할 질문

1. **depth 10 / 30 리랭킹이 hit@5를 올리는가** (주 질문)
2. **`max_length` 512 vs 1024** — 청크가 최대 2,613자(≈1,500토큰)라 512면 뒷부분이 잘린다.
   v1에서는 절단 위험이 0이었으므로(앵커가 700자 이후인 경우 0건) **이번에 새로 생긴 변수**다.
3. **`single` / `and` 분해** — v1에서 집계가 상쇄를 가렸다. relation별로 갈라 본다.
4. **hit@1** — v1에서 리랭커가 유일하게 이겼던 축(430 49% vs 검색 42%). 여전히 이기는가.

## 배경 — 헤드룸이 v1보다 줄었다

| | v1 헤드룸 | v2 헤드룸 | 건질 수 있는 문항 |
|---|---:|---:|---|
| TEST 430 | +.112 | **+.102** | 48 → 44건 |
| VAL 355 | +.164 | **+.138** | 58 → 49건 |

표 원자성 수정이 리랭커와 같은 구간을 먼저 가져갔다. v1에서 TEST hit@5가 flat(−.002)이었고
여지가 더 좁아졌으므로 **기대는 낮게 잡되, 절단(max_length)이 v1 결과를 왜곡했을 가능성은 남아 있다.**

> 주의: 베이스라인은 **실경로(라우터 질의분류)** 기준이다. `eval_anchor_v3`의 기본 경로는
> gold `business_function`을 주입하므로 VAL에서 .896으로 더 높게 나온다(라우터 일치율 71%).
> 배포에서의 리랭킹 효과를 보려면 실경로가 맞다.


## 0. 런타임

`런타임 > 런타임 유형 변경`에서 **GPU(T4 이상)** 선택.


In [ ]:
import subprocess, sys
print(sys.version)
try:
    print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total',
                                   '--format=csv,noheader'], text=True).strip())
except Exception as e:
    print('GPU 없음 — CPU로 진행(매우 느림):', type(e).__name__)


## 1. 환경 확인 — 설치하지 않는다

**torch를 업그레이드하지 말 것.** Colab은 torch/torchvision을 짝맞춰 깔아두고, torch만 올리면
`operator torchvision::nms does not exist` → `Could not import module 'XLMRobertaForSequenceClassification'`
로 깨진다(v1에서 실제로 겪음). 이미 깨진 세션이면 **런타임 > 세션 다시 시작**.


In [ ]:
import torch, transformers
print('torch       ', torch.__version__)
print('transformers', transformers.__version__)
try:
    import torchvision; print('torchvision ', torchvision.__version__, '— 짝 정상')
except Exception as e:
    print('torchvision 깨짐:', type(e).__name__, '→ 세션 다시 시작 후 torch 업그레이드 없이 재실행')
from packaging.version import Version
assert Version(transformers.__version__) >= Version('4.40'), 'transformers가 낮음 — --no-deps로 올린 뒤 세션 재시작'
print('\n설치 불필요')


## 2. 입력 업로드

```bash
cd crawler && python ../rerank_colab/prepare_inputs.py   # → rerank_colab/rerank_inputs.zip
```


In [ ]:
import json, zipfile, io, os
if not os.path.exists('chunks.jsonl'):
    from google.colab import files
    up = files.upload()
    with zipfile.ZipFile(io.BytesIO(up[next(iter(up))])) as z:
        z.extractall('.')

def load(p):
    with open(p, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

CHUNKS = {c['chunk_id']: c for c in load('chunks.jsonl')}
META = json.load(open('meta.json', encoding='utf-8'))
print('청크', META['n_chunks'], '· 길이', META['chunk_len'])
print('파라미터', META['params'])
for k, v in META['baseline'].items():
    print(f"  {k}: n={v['n_scored']} · hit@1 {v['anchor']['hit@1']:.4f} · hit@5 {v['anchor']['hit@5']:.4f}"
          f" · 천장 hit@30 {v['anchor']['hit@30']:.4f} · id폴백 {v['n_id_fallback']}")
    print(f"       relation {v['relation']}")
    print(f"       relation hit@5 {v['relation_hit5']}")


## 3. 판정 로직 — `crawler/eval_anchor_v3.py`와 동일 규약

- 앵커는 `gt_anchors` → `gt_anchor_span` 폴백, NFKC + 공백 전면 제거로 정규화
- 앵커가 덮지 않는 gold는 `chunk_id` 폴백(하이브리드)
- **코퍼스에 없는 gold(stale)는 조건에서 제외** — 재청킹으로 사라진 좌표를 필수 조건으로
  남기면 `and`가 영구 불가능해진다. v2에서 고친 부분이라 반드시 반영해야 베이스라인이 맞는다
- `and` = ALL, 그 외(`single`/`or`/`near_dup`/`duplicate`/`contained`) = ANY


In [ ]:
import re, unicodedata, collections

KS = (1, 3, 5, 10, 30)
RELATION_ANY = {'single','or','duplicate','near_dup','contained'}
RELATION_ALL = {'and'}

def norm_text(t):
    return re.sub(r'\s+', '', unicodedata.normalize('NFKC', t or ''))

NT = {cid: norm_text(c.get('text','')) for cid, c in CHUNKS.items()}

def gold_ids(it):
    ids = it.get('gt_chunk_ids') or ([it['gt_chunk_id']] if it.get('gt_chunk_id') else [])
    return list(dict.fromkeys([ids] if isinstance(ids, str) else ids))

def get_anchors(it):
    a = it.get('gt_anchors')
    if isinstance(a, str): a = [a]
    if not a:
        s = it.get('gt_anchor_span'); a = [s] if s else []
    return list(dict.fromkeys(norm_text(x) for x in a if x and x.strip()))

def relation(it):
    lab = it.get('chunk_relation') or 'single'
    if lab in RELATION_ALL: return 'and'
    if lab in RELATION_ANY: return 'any'
    raise ValueError(f"{it.get('id')}: unsupported chunk_relation={lab!r}")

def conditions(it):
    """(is_and, [조건집합...]). 집합 하나 = '이 중 하나라도 top-k면 충족'."""
    anchors = get_anchors(it)
    sets = [s for s in ({c for c in NT if a in NT[c]} for a in anchors) if s]
    covered = set().union(*sets) if sets else set()
    for c in gold_ids(it):
        if c not in NT:      # stale — 사라진 좌표는 조건에서 제외(v2 수정)
            continue
        if c not in covered and not any(a in NT[c] for a in anchors):
            sets.append({c})
    return relation(it) == 'and', sets

def score_run(items, orders):
    """→ (전체 지표, relation별 hit@5, n)"""
    m = {f'hit@{k}': 0 for k in KS}; m['mrr'] = 0.0
    per = collections.defaultdict(lambda: [0, 0]); n = 0
    for it in items:
        if it.get('answerable') is not True: continue
        is_and, sets = conditions(it)
        if not sets: continue
        n += 1; lab = it.get('chunk_relation') or 'single'; per[lab][1] += 1
        pos = {c: r for r, c in enumerate(orders[it['id']], 1)}
        firsts = [min((pos[c] for c in s if c in pos), default=None) for s in sets]
        if is_and:
            need = None if any(f is None for f in firsts) else max(firsts)
        else:
            ok = [f for f in firsts if f is not None]; need = min(ok) if ok else None
        if need is None: continue
        for k in KS:
            if need <= k: m[f'hit@{k}'] += 1
        if need <= 5: per[lab][0] += 1
        if need <= 10: m['mrr'] += 1.0 / need
    return ({k: v/n for k, v in m.items()},
            {l: (a/b, b) for l, (a, b) in per.items()}, n)


## 4. 베이스라인 재현 + assert — 통과해야 이후 수치를 믿을 수 있다


In [ ]:
SETS = {}
for name in ('430','355'):
    items = load(f'gt_natural{name}.jsonl')
    cands = {c['id']: c for c in load(f'candidates_{name}.jsonl')}
    SETS[name] = (items, cands)

ok = True
for name, (items, cands) in SETS.items():
    orders = {i: [x['chunk_id'] for x in c['candidates']] for i, c in cands.items()}
    got, per, n = score_run(items, orders)
    exp = META['baseline'][name]['anchor']
    diff = {k: round(got[k]-exp[k], 6) for k in ('hit@1','hit@3','hit@5','hit@10','mrr')
            if abs(got[k]-exp[k]) > 1e-6}
    ok &= not diff
    print(f"{name} n={n} hit@1 {got['hit@1']:.4f} hit@5 {got['hit@5']:.4f} "
          f"{'OK' if not diff else '!! 불일치 '+str(diff)}")
    print('     relation hit@5:', {k: round(v[0],4) for k, v in sorted(per.items())})
assert ok, '판정 로직이 eval_anchor_v3와 어긋난다 — 진행 금지'
print('\n판정 동치 확인 완료')


## 5. 리랭커 로드 + 절단 진단

청크가 최대 2,613자다. `max_length`가 짧으면 뒷부분이 잘려 리랭커가 근거를 못 본다.
**후보 청크 중 몇 %가 512/1024 토큰을 넘는지 먼저 센다** — 이게 v1과 달라진 핵심 변수다.


In [ ]:
import torch, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL = 'BAAI/bge-reranker-v2-m3'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL)
try:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, dtype=DTYPE)
except TypeError:
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, torch_dtype=DTYPE)
except ModuleNotFoundError as e:
    raise RuntimeError(f'{e}\n\ntorch/torchvision 짝 문제 — 세션 다시 시작 후 재실행') from e
model = model.to(DEVICE).eval()
print(MODEL, '·', DEVICE, '·', DTYPE, '· 라벨수', model.config.num_labels)

# 절단 진단 — 후보로 등장하는 청크만 대상
used = {x['chunk_id'] for _, (it, cd) in SETS.items() for c in cd.values() for x in c['candidates']}
lens = sorted(len(tok(CHUNKS[c]['text'])['input_ids']) for c in used)
import statistics
print(f"\n후보 청크 {len(used)}개 토큰수: 중앙 {statistics.median(lens):.0f} "
      f"p90 {lens[int(len(lens)*.9)]} max {lens[-1]}")
for L in (512, 1024):
    over = sum(1 for x in lens if x > L)
    print(f"  max_length={L}: 절단되는 청크 {over}/{len(lens)} ({over/len(lens):.1%})")


## 6. 리랭킹 — `max_length` 512 / 1024 두 번

top-30 전량을 두 설정으로 채점해 캐시한다. 785문항 × 30 × 2 = 약 47k 쌍.
T4에서 512는 5~8분, 1024는 10~15분 예상.


In [ ]:
@torch.no_grad()
def rerank_scores(query, chunk_ids, max_length, batch_size=16):
    texts = [CHUNKS[c]['text'] for c in chunk_ids]
    out = []
    for i in range(0, len(texts), batch_size):
        b = texts[i:i+batch_size]
        enc = tok([query]*len(b), b, padding=True, truncation=True,
                  max_length=max_length, return_tensors='pt').to(DEVICE)
        out += model(**enc).logits.view(-1).float().cpu().tolist()
    return out

RR = {}          # {max_length: {set: {id: {chunk_id: score}}}}
for ML in (512, 1024):
    RR[ML] = {}
    for name, (items, cands) in SETS.items():
        store = {}; t0 = time.perf_counter()
        for k, it in enumerate(items, 1):
            c = cands[it['id']]
            ids = [x['chunk_id'] for x in c['candidates']]
            store[it['id']] = dict(zip(ids, rerank_scores(c['question'], ids, ML)))
            if k % 150 == 0: print(f'  ML={ML} {name} {k}/{len(items)} · {time.perf_counter()-t0:.0f}s')
        RR[ML][name] = store
        print(f'ML={ML} {name} 완료 {time.perf_counter()-t0:.0f}s')

json.dump({str(k): v for k, v in RR.items()}, open('rerank_scores_v2.json','w'), ensure_ascii=False)
print('저장: rerank_scores_v2.json — 내려두면 GPU 없이 로컬 재분석 가능')


## 7. 스윕 — depth × blend × max_length × post_cap

- **depth** 10 / 30 (요청) — depth 밖 후보는 원래 순서로 뒤에 붙는다
- **blend `w`** — `w × 리랭커z + (1-w) × 검색z`. 각각 z-정규화(검색 점수는 RRF라 스케일이 다르다).
  `w=0`이 베이스라인이므로 **`w=0` 행이 4번 셀 값과 같아야 한다**(자체 검증)
- **max_length** 512 / 1024
- **post_cap** 0(없음) / 5 — 리랭킹 **뒤** 문서당 상한. `BF_DOC_CAP`은 K≤5에서 무효라
  (HOLES H29) 여기서만 의미가 생긴다


In [ ]:
import statistics
def zscore(xs):
    if len(xs) < 2: return [0.0]*len(xs)
    m, s = statistics.mean(xs), statistics.pstdev(xs)
    return [0.0]*len(xs) if s == 0 else [(x-m)/s for x in xs]

DOC_OF = {cid: c['parent_doc_id'] for cid, c in CHUNKS.items()}

def reorder(cand, rr, depth, w, post_cap):
    head = cand['candidates'][:depth]
    tail = [x['chunk_id'] for x in cand['candidates'][depth:]]
    zr = zscore([rr[x['chunk_id']] for x in head])
    zs = zscore([x['score'] for x in head])
    fused = sorted(zip((x['chunk_id'] for x in head),
                       (w*a + (1-w)*b for a, b in zip(zr, zs))), key=lambda t: -t[1])
    order = [c for c, _ in fused]
    if post_cap:
        seen, keep = {}, []
        for c in order:
            d = DOC_OF[c]
            if seen.get(d, 0) < post_cap:
                keep.append(c); seen[d] = seen.get(d, 0) + 1
        kept = set(keep); order = keep + [c for c in order if c not in kept]
    return order + tail

DEPTHS = (10, 30); WS = (0.0, 0.2, 0.3, 0.5, 0.7, 1.0); CAPS = (0, 5)
rows = []
for ML in (512, 1024):
    for name, (items, cands) in SETS.items():
        base = META['baseline'][name]['anchor']
        for depth in DEPTHS:
            for w in WS:
                for cap in CAPS:
                    orders = {i: reorder(c, RR[ML][name][i], depth, w, cap)
                              for i, c in cands.items()}
                    m, per, n = score_run(items, orders)
                    rows.append(dict(ml=ML, set=name, depth=depth, w=w, cap=cap, n=n,
                                     **{k: round(m[k],4) for k in ('hit@1','hit@3','hit@5','hit@10','mrr')},
                                     d5=round(m['hit@5']-base['hit@5'], 4),
                                     d1=round(m['hit@1']-base['hit@1'], 4),
                                     **{f'r_{l}': round(v[0],4) for l, v in per.items()}))
        print(f'ML={ML} {name} 스윕 완료')
json.dump(rows, open('sweep_v2.json','w'), ensure_ascii=False, indent=1)
print(len(rows), 'configs')

# 자체 검증 — w=0은 베이스라인과 같아야 한다(post_cap=0 기준)
for name in ('430','355'):
    b = META['baseline'][name]['anchor']['hit@5']
    z = [r for r in rows if r['set']==name and r['w']==0.0 and r['cap']==0]
    assert all(abs(r['hit@5']-b) < 1e-6 for r in z), f'{name}: w=0이 베이스라인과 다르다'
print('w=0 == 베이스라인 확인')


### 결과 — 주 질문: depth 10/30이 hit@5를 올리는가


In [ ]:
def show(name, ml, k=12):
    b = META['baseline'][name]['anchor']
    print(f"[{name}] ML={ml} · 베이스라인 hit@1 {b['hit@1']:.4f} hit@5 {b['hit@5']:.4f}")
    r = sorted((x for x in rows if x['set']==name and x['ml']==ml), key=lambda x: -x['hit@5'])
    print(f"{'depth':>6}{'w':>5}{'cap':>4}{'hit@1':>8}{'hit@3':>8}{'hit@5':>8}{'hit@10':>8}"
          f"{'mrr':>8}{'Δhit@5':>9}{'Δhit@1':>9}")
    for x in r[:k]:
        print(f"{x['depth']:>6}{x['w']:>5}{x['cap']:>4}{x['hit@1']:>8.4f}{x['hit@3']:>8.4f}"
              f"{x['hit@5']:>8.4f}{x['hit@10']:>8.4f}{x['mrr']:>8.4f}{x['d5']:>+9.4f}{x['d1']:>+9.4f}")
    print()

for ml in (512, 1024):
    for n in ('430','355'):
        show(n, ml)


### max_length 효과 · relation 분해 · 두 셋 전이성

세 표를 나란히 본다. **한쪽 셋만 좋은 설정은 버린다** — 파라미터 튜닝에서 이미 겪었다.


In [ ]:
print('=== max_length 512 vs 1024 (같은 depth·w·cap) ===')
key = lambda x: (x['set'], x['depth'], x['w'], x['cap'])
A = {key(x): x for x in rows if x['ml']==512}
B = {key(x): x for x in rows if x['ml']==1024}
best = sorted(set(A) & set(B), key=lambda k: -(B[k]['hit@5']-A[k]['hit@5']))
print(f"{'(set,depth,w,cap)':<24}{'512':>9}{'1024':>9}{'Δ':>9}")
for k in best[:8]:
    print(f"{str(k):<24}{A[k]['hit@5']:>9.4f}{B[k]['hit@5']:>9.4f}{B[k]['hit@5']-A[k]['hit@5']:>+9.4f}")
d = [B[k]['hit@5']-A[k]['hit@5'] for k in set(A)&set(B)]
print(f"  전체 평균 Δ(1024−512) {statistics.mean(d):+.4f} · 양수 {sum(1 for x in d if x>0)}/{len(d)}")

print('\n=== relation 분해 (430, ML별 최적 w) ===')
for ml in (512, 1024):
    r = sorted((x for x in rows if x['set']=='430' and x['ml']==ml), key=lambda x: -x['hit@5'])[0]
    b = META['baseline']['430']['relation_hit5']
    print(f"  ML={ml} depth={r['depth']} w={r['w']} cap={r['cap']}")
    for lab in ('single','and','or'):
        if f'r_{lab}' in r:
            print(f"    {lab:<8} base {b.get(lab,0):.4f} → rr {r[f'r_{lab}']:.4f} "
                  f"({r[f'r_{lab}']-b.get(lab,0):+.4f})")

print('\n=== 두 셋 전이성 (min hit@5 최대화) ===')
kk = lambda x: (x['ml'], x['depth'], x['w'], x['cap'])
P = {kk(x): x for x in rows if x['set']=='430'}
Q = {kk(x): x for x in rows if x['set']=='355'}
joint = sorted(set(P)&set(Q), key=lambda k: -min(P[k]['hit@5'], Q[k]['hit@5']))
print(f"{'(ml,depth,w,cap)':<24}{'430':>9}{'355':>9}{'min':>9}{'Δ430':>9}{'Δ355':>9}")
for k in joint[:10]:
    print(f"{str(k):<24}{P[k]['hit@5']:>9.4f}{Q[k]['hit@5']:>9.4f}"
          f"{min(P[k]['hit@5'],Q[k]['hit@5']):>9.4f}{P[k]['d5']:>+9.4f}{Q[k]['d5']:>+9.4f}")


## 8. 결과 내려받기


In [ ]:
from google.colab import files
files.download('rerank_scores_v2.json')
files.download('sweep_v2.json')


## 판정 규율

1. **주지표는 hit@5.** `w=0` 행이 베이스라인과 일치하는지 7번 셀 assert로 확인된다.
2. **한쪽 셋만 좋은 설정은 버린다.** TEST 430 단독 최적이 VAL을 깎으면 채택 불가
   (`DECISIONS.md` 08-14 ④에서 같은 함정을 겪었다).
3. **`hit@30`은 리랭킹으로 변하지 않는다** — 집합은 그대로고 순서만 바뀐다.
   천장은 TEST .9977 / VAL .9803이며 넘으려면 검색 단계를 손봐야 한다.
4. **`and`(430에서 55건)와 `single`을 반드시 갈라 본다.** v1에서 집계가 상쇄를 가렸다.
5. **최적점이 경계(depth 30 · w 1.0 · cap 0)에 붙으면 범위를 넓혀 재실행.**
6. hit@5가 두 셋에서 안 오르면 **미채택 유지가 맞다.** v1에서도 TEST는 flat이었고
   헤드룸은 그때보다 줄었다. `max_length` 1024가 그걸 뒤집는지가 이번 실험의 핵심이다.
